# Making chloropleth maps in Altair

Here's a quick example of how to make a chloropleth map in Altair.  In this example, we'll work with a fairly large data set of baby names in France from 1900-2019, broken down by department.

To work with geographical data, we'll use the `geopandas`, which loads `pandas` dataframes, but with support for geographical outlines in the `geojson` format.  You can use these dataframes just as you would a regular `pandas` dataframe, but they will include that extra geographical outline data.

To get started, we'll need to import our libraries.

In [3]:
import altair as alt
import pandas as pd
import geopandas as gpd # Requires geopandas -- e.g.: conda install -c conda-forge geopandas
alt.data_transformers.enable('json') # Let Altair/Vega-Lite work with large data sets

pass

# Reading our names data

Now, let's read in our dataset.  The exported data is in CSV format, but with a `;` separator instead of commas.  The INSEE data collapses rare names or where department-level information has been elided (presumably to protect individuals with uncommon names or who were one of the only ones born with that name in a given year).  We'll strip those out.

In [4]:
names = pd.read_csv("dpt2020.csv", sep=";")
names.drop(names[names.preusuel == '_PRENOMS_RARES'].index, inplace=True)
names.drop(names[names.dpt == 'XX'].index, inplace=True)

names.sample(5)

,sexe,preusuel,annais,dpt,nombre
2413594,2,EVE,1985,78,4
337649,1,CYRIAQUE,1970,50,7
2943538,2,LUDIVINE,1998,07,4
482490,1,ÉTHAN,2016,56,3
2573357,2,HÉLÈNE,1964,20,8


# Loading map data

Next, let's load some map data of regions in France using `geopandas`.  These map data come from the [INSEE] and [IGN] and were processed into the `geojson` format we'll need to work with by [Grégoire David].  Here's the [github] repository.

In this example, we'll work with the simplified departments tiles for the Hexagon, but that repository contains higher-resolution versions, the DOM-TOM, and more.

[Grégoire David]: https://gregoiredavid.fr
[INSEE]: http://www.insee.fr/fr/methodes/nomenclatures/cog/telechargement.asp
[IGN]: https://geoservices.ign.fr/adminexpress
[github]: https://github.com/gregoiredavid/france-geojson/

In [5]:
depts = gpd.read_file('departements-version-simplifiee.geojson')

depts.sample(5)

,code,nom,geometry
13,14,Calvados,"POLYGON ((-1.11962 49.35557, -1.07822 49.38849..."
22,24,Dordogne,"POLYGON ((0.62974 45.71457, 0.65423 45.6887, 0..."
11,12,Aveyron,"POLYGON ((2.20748 44.61553, 2.20841 44.64384, ..."
7,08,Ardennes,"POLYGON ((4.23316 49.95775, 4.3081 49.96952, 4..."
42,42,Loire,"POLYGON ((3.89953 46.27591, 3.9094 46.25773, 3..."


Notice how `depts` is a geopandas dataframe.  We'll use it just as a regular `pandas` dataframe, but it includes the geometry info we need to be able to draw those regions when we pass them into Altair.  We just need to make sure that when we work with our data, we keep them in a geopandas dataframe and not a plain dataframe if we want to draw the departments.

In the next cell, notice how we do a right-merge to bring in department data into names.  We do this as a merge on `depts` because we need a geopandas dataframe.  Remember, `depts` is a geopandas dataframe, while `names` is a regular dataframe.  If we did a left merge on `names`, we'd end up with a regular pandas dataframe. After this merge, both `names` and `depts` will be geopandas dataframes.

**Hint:** Be careful when you do your data joins here.  It's easy to accidentally merge the wrong way to accidentally create a _much bigger_ dataset.

In [6]:
# Keep a reference around to the plain pandas dataframe, without geometry data, just in case
just_names = names

names = depts.merge(names, how='right', left_on='code', right_on='dpt')

names.sample(5)

,code,nom,geometry,sexe,preusuel,annais,dpt,nombre
2924281,91,Essonne,"POLYGON ((2.22656 48.7761, 2.23298 48.7662, 2....",2,MAELINE,2005,91,3
1574909,22,Côtes-d'Armor,"POLYGON ((-3.65914 48.65921, -3.63649 48.67069...",1,TIMOTHÉE,1994,22,3
3041676,74,Haute-Savoie,"POLYGON ((6.80252 45.77837, 6.75551 45.76635, ...",2,MARIE-JOSÉ,1964,74,4
245403,83,Var,"MULTIPOLYGON (((6.4348 43.01554, 6.4552 43.026...",1,BRYAN,1996,83,14
2217285,92,Hauts-de-Seine,"POLYGON ((2.29097 48.95097, 2.32697 48.94536, ...",2,DELPHINE,2000,92,17


# Show a name over all years

Now we'll choose a name to show across all years.  To that, we'll group all of the names in a department together (squashing the years together) and use the sum.

In [7]:
grouped = names.groupby(['dpt', 'preusuel', 'sexe'], as_index=False).sum(numeric_only=True)
grouped = depts.merge(grouped, how='right', left_on='code', right_on='dpt') # Add geometry data back in
grouped

,code,nom,geometry,dpt,preusuel,sexe,nombre
0,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,AARON,1,160
1,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,ABBY,2,3
2,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,ABDALLAH,1,7
3,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,ABDEL,1,3
4,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,ABDELKADER,1,3
...,...,...,...,...,...,...,...
239574,NaN,NaN,None,974,ÉSAÏE,1,3
239575,NaN,NaN,None,974,ÉTHAN,1,53
239576,NaN,NaN,None,974,ÉTIENNE,1,3
239577,NaN,NaN,None,974,ÉVA,2,32


Now let's pick a name and check out how it's distribution over the last 120 years across Metropolitan France.  In this example, I choose the name “Lucien,” which I rather like for some reason.

In [ ]:
name = 'LUCIEN'
subset = grouped[grouped.preusuel == name]
alt.Chart(subset).mark_geoshape(stroke='white').encode(
    tooltip=['nom', 'code', 'nombre'],
    color='nombre',
).properties(width=800, height=600)

In [12]:
names04 = names[names.annais == '2004']
grouped = names04.groupby(['preusuel', 'sexe'], as_index=False).sum(numeric_only=True)


options = [1, 2]
labels = ['Garçon', 'Fille']

input_dropdown = alt.binding_radio(
    # Add the empty selection which shows all when clicked
    options=options + [None],
    labels=labels + ['Mixte'],
    name='Sexe: '
)
selection = alt.selection_point(
    fields=['sexe'],
    bind=input_dropdown
)

top_20_2004 = alt.Chart(grouped).mark_bar().encode(
    x = alt.X('nombre:Q', title='Nombre de naissances'),
    y = alt.Y('preusuel', sort='-x', title='Prénom'),
    color=alt.Color(
        'sexe:N', 
        legend=alt.Legend(title="Sexe", labelExpr="datum.value == '1' ? 'Garçon' : 'Fille'"), 
        scale=alt.Scale(domain=[1,2], range=['steelblue', '#F2A6A0'])
    ),
).add_params(
    selection
).transform_filter(
    selection
).transform_window(
    rank='rank(nombre)',
    sort=[alt.SortField('nombre', order='descending')]
).transform_filter(
    alt.datum.rank <= 20,
)


top_20_2004


alt.Chart(...)